# Experiment 01: Ticket Group Size

This experiment tests whether the number of passengers sharing a ticket improves a Decision Tree's Titanic survival predictions. The final decision is based on Stratified 5-Fold Cross-Validation, not the Kaggle leaderboard.

## Hypothesis

Passengers travelling with more people under the same ticket may have a better chance of survival because they could support each other during evacuation.

In [1]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

train_df = pd.read_csv("../../data/train.csv")

In [2]:
def group_family(size):
    if size == 1:
        return "Alone"
    if 2 <= size <= 4:
        return "Small"
    return "Large"


def add_base_features(df):
    df = df.copy()
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["FamilyGroup"] = df["FamilySize"].apply(group_family)

    df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
    df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    rare_titles = ["Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"]
    df["Title"] = df["Title"].replace(rare_titles, "Rare")
    return df


def add_ticket_group_size(df):
    df = df.copy()
    df["TicketGroupSize"] = df.groupby("Ticket")["Ticket"].transform("count")
    return df

In [3]:
base_features = ["Pclass", "Sex", "Age", "Fare", "Embarked", "FamilyGroup", "Title"]
ticket_features = base_features + ["TicketGroupSize"]

numeric_base = ["Pclass", "Age", "Fare"]
numeric_ticket = numeric_base + ["TicketGroupSize"]
categorical_features = ["Sex", "Embarked", "FamilyGroup", "Title"]

def make_pipeline(numeric_features):
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ])
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", DecisionTreeClassifier(max_depth=3, random_state=42)),
    ])

base_df = add_base_features(train_df)
ticket_df = add_ticket_group_size(base_df)

X_base = base_df[base_features]
X_ticket = ticket_df[ticket_features]
y = train_df["Survived"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [4]:
baseline_scores = cross_val_score(make_pipeline(numeric_base), X_base, y, cv=cv, scoring="accuracy")
ticket_scores = cross_val_score(make_pipeline(numeric_ticket), X_ticket, y, cv=cv, scoring="accuracy")

results = pd.DataFrame({
    "Experiment": ["Baseline", "Baseline + TicketGroupSize"],
    "Mean CV accuracy": [baseline_scores.mean(), ticket_scores.mean()],
    "CV standard deviation": [baseline_scores.std(), ticket_scores.std()],
})

print(results.round(3))

                   Experiment  Mean CV accuracy  CV standard deviation
0                    Baseline             0.826                  0.013
1  Baseline + TicketGroupSize             0.824                  0.010


## Result and Decision

The baseline achieved **0.826 +/- 0.013** mean CV accuracy. Adding `TicketGroupSize` achieved **0.824 +/- 0.010**.

`TicketGroupSize` is not included in the final model because it did not improve the mean Cross-Validation accuracy.